# Investigating LATERAL JOIN Patterns

**SOLUTIONS NOTEBOOK**

## What is a LATERAL join?

`LATERAL` (SQL:1999, widely used in **PostgreSQL**, also in Oracle, SQL Server as `CROSS/OUTER APPLY`) lets a subquery in the `FROM` clause **reference columns from preceding tables**.

```sql
-- PostgreSQL / standard SQL
SELECT c.CustomerId, c.LastName, inv.InvoiceId, inv.Total
FROM customers c
CROSS JOIN LATERAL (
    SELECT InvoiceId, Total
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId      -- references outer table
    ORDER BY Total DESC
    LIMIT 1
) inv;
```

Without `LATERAL`, the subquery in `FROM` is evaluated independently and cannot see `c.CustomerId`.

## Common use cases for LATERAL

| Use case | Description |
|----------|-------------|
| Top-N per group | “For each customer, the 3 most expensive invoices” |
| Correlated existence | “Customers who have at least one invoice over $20” (can also use EXISTS) |
| Dependent calculations | “For each row, call a set-returning function that needs its values” |
| Nearest neighbor / “best match” | “For each employee, the nearest office” |

## Important reality for this environment

> **SQLite does not support `LATERAL` joins.**

Attempting the syntax produces a syntax error.  
We will therefore:

1. Demonstrate the failure
2. Learn the **equivalent patterns that work in SQLite**
3. Practice those patterns on the Chinook database

## SQLite equivalents you will master

| Goal | SQLite technique |
|------|------------------|
| Top-N per group | `ROW_NUMBER() OVER (PARTITION BY …)` + outer filter |
| Scalar correlated value | Correlated scalar subquery in `SELECT` |
| “Exists related rows” | `EXISTS` / `IN` / `JOIN` |
| Complex per-row set | CTE + window, or correlated subquery |


## Exercise 1 – Observe the LATERAL failure

**Goal:** Confirm that SQLite rejects `LATERAL`.

### Instructions

Run this query (it is expected to fail):

```sql
SELECT c.CustomerId, c.LastName, inv.Total
FROM customers c,
LATERAL (
    SELECT Total
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
    ORDER BY Total DESC
    LIMIT 1
) inv
LIMIT 5;
```

Note the error message.


In [ ]:
-- INTENTIONALLY INVALID in SQLite
SELECT c.CustomerId, c.LastName, inv.Total
FROM customers c,
LATERAL (
    SELECT Total
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
    ORDER BY Total DESC
    LIMIT 1
) inv
LIMIT 5;


**Solution**

```sql
-- INTENTIONALLY INVALID in SQLite
SELECT c.CustomerId, c.LastName, inv.Total
FROM customers c,
LATERAL (
    SELECT Total
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
    ORDER BY Total DESC
    LIMIT 1
) inv
LIMIT 5;
```

> This query is expected to raise a syntax error in SQLite.

**Expected outcome**
- Syntax error near `LATERAL` or near `SELECT`.
- This is normal — SQLite simply does not implement the feature.


## Exercise 2 – Top-1 per group with window functions (classic replacement)

**Business question:**  
For every customer, return their **single most expensive invoice**.

### Instructions

1. Create a CTE that selects from `invoices` and adds  
   `ROW_NUMBER() OVER (PARTITION BY CustomerId ORDER BY Total DESC) AS rn`
2. Outer query keeps only `rn = 1`
3. Join back to `customers` if you want the customer name
4. Order by customer id, limit to 10 rows for readability


In [ ]:
WITH ranked AS (
    SELECT
        CustomerId,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerId
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    r.InvoiceId,
    r.Total
FROM ranked r
JOIN customers c ON c.CustomerId = r.CustomerId
WHERE r.rn = 1
ORDER BY c.CustomerId
LIMIT 10;


**Solution**

```sql
WITH ranked AS (
    SELECT
        CustomerId,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerId
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    r.InvoiceId,
    r.Total
FROM ranked r
JOIN customers c ON c.CustomerId = r.CustomerId
WHERE r.rn = 1
ORDER BY c.CustomerId
LIMIT 10;
```

**Hints**
```sql
WITH ranked AS (
    SELECT
        CustomerId,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerId
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT c.CustomerId, c.FirstName, c.LastName,
       r.InvoiceId, r.Total
FROM ranked r
JOIN customers c ON c.CustomerId = r.CustomerId
WHERE r.rn = 1
ORDER BY c.CustomerId
LIMIT 10;
```
This is the most common and efficient replacement for a LATERAL top-1 join.


## Exercise 3 – Top-N per group (N = 2)

**Business question:**  
For each billing country, show the **two highest** invoice totals.

### Instructions

Same window technique as Exercise 2, but keep `rn <= 2`.  
Return country, invoice id, total, and the row number.


In [ ]:
WITH ranked AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT *
FROM ranked
WHERE rn <= 2
ORDER BY BillingCountry, rn;


**Solution**

```sql
WITH ranked AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT *
FROM ranked
WHERE rn <= 2
ORDER BY BillingCountry, rn;
```

**Hints**
```sql
WITH ranked AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT *
FROM ranked
WHERE rn <= 2
ORDER BY BillingCountry, rn;
```


## Exercise 4 – Correlated scalar subquery

**Business question:**  
For each customer, show the date of their **most recent** invoice.

### Instructions

Write a query against `customers` that includes a scalar subquery:

```sql
(SELECT MAX(InvoiceDate) FROM invoices i WHERE i.CustomerId = c.CustomerId)
```

Return CustomerId, FirstName, LastName, and that latest date.  
Order by CustomerId and show the first 10 rows.


In [ ]:
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    (SELECT MAX(InvoiceDate)
     FROM invoices i
     WHERE i.CustomerId = c.CustomerId) AS LastInvoiceDate
FROM customers c
ORDER BY c.CustomerId
LIMIT 10;


**Solution**

```sql
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    (SELECT MAX(InvoiceDate)
     FROM invoices i
     WHERE i.CustomerId = c.CustomerId) AS LastInvoiceDate
FROM customers c
ORDER BY c.CustomerId
LIMIT 10;
```

**Hints**
- The subquery is correlated — it references `c.CustomerId` from the outer query.
- This pattern is the natural replacement when you need a **single value** per outer row (not a set of rows).


## Exercise 5 – EXISTS (another LATERAL use-case replacement)

**Business question:**  
List customers who have **at least one** invoice with Total > 15.

### Instructions

Use `EXISTS` with a correlated subquery.  
Return CustomerId, FirstName, LastName, Country.


In [ ]:
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    c.Country
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
      AND i.Total > 15
)
ORDER BY c.CustomerId;


**Solution**

```sql
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    c.Country
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
      AND i.Total > 15
)
ORDER BY c.CustomerId;
```

**Hints**
```sql
SELECT c.CustomerId, c.FirstName, c.LastName, c.Country
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM invoices i
    WHERE i.CustomerId = c.CustomerId
      AND i.Total > 15
)
ORDER BY c.CustomerId;
```
`EXISTS` stops at the first matching row — usually very efficient.


## Exercise 6 – “Best match” per outer row (support rep’s top customer)

**Business question:**  
For each support representative, who is their **highest-revenue customer**?

### Instructions

1. Build a CTE that, for every (SupportRepId, CustomerId) pair, computes total revenue and  
   assigns `ROW_NUMBER() OVER (PARTITION BY SupportRepId ORDER BY revenue DESC)`
2. Keep `rn = 1`
3. Join to `employees` to get the rep’s name and to `customers` for the customer name


In [ ]:
WITH customer_rev AS (
    SELECT
        c.SupportRepId,
        c.CustomerId,
        SUM(i.Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY c.SupportRepId
            ORDER BY SUM(i.Total) DESC
        ) AS rn
    FROM customers c
    JOIN invoices i ON c.CustomerId = i.CustomerId
    GROUP BY c.SupportRepId, c.CustomerId
)
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS RepName,
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS CustomerName,
    cr.Revenue
FROM customer_rev cr
JOIN employees e ON e.EmployeeId = cr.SupportRepId
JOIN customers c ON c.CustomerId = cr.CustomerId
WHERE cr.rn = 1
ORDER BY e.EmployeeId;


**Solution**

```sql
WITH customer_rev AS (
    SELECT
        c.SupportRepId,
        c.CustomerId,
        SUM(i.Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY c.SupportRepId
            ORDER BY SUM(i.Total) DESC
        ) AS rn
    FROM customers c
    JOIN invoices i ON c.CustomerId = i.CustomerId
    GROUP BY c.SupportRepId, c.CustomerId
)
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS RepName,
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS CustomerName,
    cr.Revenue
FROM customer_rev cr
JOIN employees e ON e.EmployeeId = cr.SupportRepId
JOIN customers c ON c.CustomerId = cr.CustomerId
WHERE cr.rn = 1
ORDER BY e.EmployeeId;
```

**Hints**
```sql
WITH customer_rev AS (
    SELECT
        c.SupportRepId,
        c.CustomerId,
        SUM(i.Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY c.SupportRepId
            ORDER BY SUM(i.Total) DESC
        ) AS rn
    FROM customers c
    JOIN invoices i ON c.CustomerId = i.CustomerId
    GROUP BY c.SupportRepId, c.CustomerId
)
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS RepName,
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS CustomerName,
    cr.Revenue
FROM customer_rev cr
JOIN employees e ON e.EmployeeId = cr.SupportRepId
JOIN customers c ON c.CustomerId = cr.CustomerId
WHERE cr.rn = 1
ORDER BY e.EmployeeId;
```


## Exercise 7 – Lateral-style “apply a limit per row” with a join condition

**Business question:**  
For each genre, return the **longest track** (by milliseconds).

### Instructions

Use the window-function pattern again.  
Join `tracks` and `genres`, partition by genre, order by milliseconds descending, keep rn = 1.


In [ ]:
WITH ranked AS (
    SELECT
        g.Name AS Genre,
        t.TrackId,
        t.Name AS Track,
        t.Milliseconds,
        ROW_NUMBER() OVER (
            PARTITION BY g.GenreId
            ORDER BY t.Milliseconds DESC
        ) AS rn
    FROM tracks t
    JOIN genres g ON t.GenreId = g.GenreId
)
SELECT Genre, TrackId, Track, Milliseconds
FROM ranked
WHERE rn = 1
ORDER BY Milliseconds DESC;


**Solution**

```sql
WITH ranked AS (
    SELECT
        g.Name AS Genre,
        t.TrackId,
        t.Name AS Track,
        t.Milliseconds,
        ROW_NUMBER() OVER (
            PARTITION BY g.GenreId
            ORDER BY t.Milliseconds DESC
        ) AS rn
    FROM tracks t
    JOIN genres g ON t.GenreId = g.GenreId
)
SELECT Genre, TrackId, Track, Milliseconds
FROM ranked
WHERE rn = 1
ORDER BY Milliseconds DESC;
```

**Hints**
```sql
WITH ranked AS (
    SELECT
        g.Name AS Genre,
        t.TrackId,
        t.Name AS Track,
        t.Milliseconds,
        ROW_NUMBER() OVER (
            PARTITION BY g.GenreId
            ORDER BY t.Milliseconds DESC
        ) AS rn
    FROM tracks t
    JOIN genres g ON t.GenreId = g.GenreId
)
SELECT Genre, TrackId, Track, Milliseconds
FROM ranked
WHERE rn = 1
ORDER BY Milliseconds DESC;
```


## Exercise 8 – Combining correlated subquery + aggregation

**Business question:**  
Show each employee together with:
- the number of customers they support
- the total revenue generated by those customers

### Instructions

You can solve this either with JOINs + GROUP BY or with correlated scalar subqueries.  
Try the **correlated subquery** style to stay close to the LATERAL mindset.


In [ ]:
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS Name,
    (SELECT COUNT(*)
     FROM customers c
     WHERE c.SupportRepId = e.EmployeeId) AS CustomerCount,
    (SELECT COALESCE(SUM(i.Total), 0)
     FROM customers c
     JOIN invoices i ON i.CustomerId = c.CustomerId
     WHERE c.SupportRepId = e.EmployeeId) AS TotalRevenue
FROM employees e
WHERE e.Title LIKE '%Support%' OR e.Title LIKE '%Sales%'
ORDER BY TotalRevenue DESC;


**Solution**

```sql
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS Name,
    (SELECT COUNT(*)
     FROM customers c
     WHERE c.SupportRepId = e.EmployeeId) AS CustomerCount,
    (SELECT COALESCE(SUM(i.Total), 0)
     FROM customers c
     JOIN invoices i ON i.CustomerId = c.CustomerId
     WHERE c.SupportRepId = e.EmployeeId) AS TotalRevenue
FROM employees e
WHERE e.Title LIKE '%Support%' OR e.Title LIKE '%Sales%'
ORDER BY TotalRevenue DESC;
```

**Hints (correlated style)**
```sql
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS Name,
    (SELECT COUNT(*)
     FROM customers c
     WHERE c.SupportRepId = e.EmployeeId) AS CustomerCount,
    (SELECT COALESCE(SUM(i.Total), 0)
     FROM customers c
     JOIN invoices i ON i.CustomerId = c.CustomerId
     WHERE c.SupportRepId = e.EmployeeId) AS TotalRevenue
FROM employees e
WHERE e.Title LIKE '%Support%' OR e.Title LIKE '%Sales%'
ORDER BY TotalRevenue DESC;
```


## Exercise 9 – Anti-pattern awareness: when NOT to force LATERAL thinking

**Question:**  
“List every invoice together with its customer’s country.”

This is a simple equi-join. Writing it with a correlated subquery or a LATERAL-style pattern would be unnecessary and slower.

### Instructions

Write the clean, idiomatic JOIN version.


In [ ]:
SELECT
    i.InvoiceId,
    i.Total,
    c.FirstName,
    c.LastName,
    c.Country
FROM invoices i
JOIN customers c ON c.CustomerId = i.CustomerId
ORDER BY i.InvoiceId
LIMIT 15;


**Solution**

```sql
SELECT
    i.InvoiceId,
    i.Total,
    c.FirstName,
    c.LastName,
    c.Country
FROM invoices i
JOIN customers c ON c.CustomerId = i.CustomerId
ORDER BY i.InvoiceId
LIMIT 15;
```

**Hints**
```sql
SELECT
    i.InvoiceId,
    i.Total,
    c.FirstName,
    c.LastName,
    c.Country
FROM invoices i
JOIN customers c ON c.CustomerId = i.CustomerId
ORDER BY i.InvoiceId
LIMIT 15;
```
Use the simplest tool that solves the problem.


## Exercise 10 – Challenge: Top country per year + that country’s top invoice

**Business question:**  
For each year:
1. Find the billing country with the highest total revenue that year
2. Also show the single highest invoice that belongs to that country in that year

### Instructions

This requires two levels of “top-1” logic.  
A clean approach is nested CTEs (or window functions applied twice).


In [ ]:
WITH country_year AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS country_rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
),
top_countries AS (
    SELECT Year, BillingCountry, Revenue
    FROM country_year
    WHERE country_rn = 1
),
ranked_invoices AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate), BillingCountry
            ORDER BY Total DESC
        ) AS inv_rn
    FROM invoices
)
SELECT
    tc.Year,
    tc.BillingCountry,
    tc.Revenue AS CountryRevenue,
    ri.InvoiceId AS TopInvoiceId,
    ri.Total AS TopInvoiceTotal
FROM top_countries tc
JOIN ranked_invoices ri
  ON ri.Year = tc.Year
 AND ri.BillingCountry = tc.BillingCountry
 AND ri.inv_rn = 1
ORDER BY tc.Year;


**Solution**

```sql
WITH country_year AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS country_rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
),
top_countries AS (
    SELECT Year, BillingCountry, Revenue
    FROM country_year
    WHERE country_rn = 1
),
ranked_invoices AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate), BillingCountry
            ORDER BY Total DESC
        ) AS inv_rn
    FROM invoices
)
SELECT
    tc.Year,
    tc.BillingCountry,
    tc.Revenue AS CountryRevenue,
    ri.InvoiceId AS TopInvoiceId,
    ri.Total AS TopInvoiceTotal
FROM top_countries tc
JOIN ranked_invoices ri
  ON ri.Year = tc.Year
 AND ri.BillingCountry = tc.BillingCountry
 AND ri.inv_rn = 1
ORDER BY tc.Year;
```

**Hints (one possible shape)**
```sql
WITH country_year AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS country_rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
),
top_countries AS (
    SELECT Year, BillingCountry, Revenue
    FROM country_year
    WHERE country_rn = 1
),
ranked_invoices AS (
    SELECT
        strftime('%Y', i.InvoiceDate) AS Year,
        i.BillingCountry,
        i.InvoiceId,
        i.Total,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', i.InvoiceDate), i.BillingCountry
            ORDER BY i.Total DESC
        ) AS inv_rn
    FROM invoices i
)
SELECT
    tc.Year,
    tc.BillingCountry,
    tc.Revenue AS CountryRevenue,
    ri.InvoiceId AS TopInvoiceId,
    ri.Total AS TopInvoiceTotal
FROM top_countries tc
JOIN ranked_invoices ri
  ON ri.Year = tc.Year
 AND ri.BillingCountry = tc.BillingCountry
 AND ri.inv_rn = 1
ORDER BY tc.Year;
```


## Summary – LATERAL mental model → SQLite toolbox

| You wish you could write… | Write this in SQLite instead |
|---------------------------|------------------------------|
| `CROSS JOIN LATERAL (… LIMIT 1)` for top-1 | `ROW_NUMBER() OVER (PARTITION BY …) … WHERE rn = 1` |
| `CROSS JOIN LATERAL (… LIMIT N)` for top-N | same window pattern with `rn <= N` |
| Scalar value depending on outer row | Correlated scalar subquery in the SELECT list |
| “Keep outer row only if related rows exist” | `EXISTS (correlated subquery)` |
| Complex per-row table expression | CTE + window functions, or multiple correlated subqueries |

### When you move to PostgreSQL / BigQuery / Snowflake later

You will be able to write true `LATERAL` (or `CROSS APPLY` / `QUALIFY`) and the mental model you built here transfers directly. The window-function and correlated-subquery patterns remain useful everywhere.
